# EurecomGPT — Phase 3: MapReduce & Spark for the RAG corpus

Run in **Google Colab** (CPU runtime is fine). Phase 3 builds a TF-IDF index two ways. The **MapReduce** half already ran — in Cloud Shell, as Cloud Run function tasks orchestrated by Cloud Workflows (`run_mr.py`, TASKS.md Tasks 3–6). This notebook picks up its result, then runs the **Spark** stages you wrote in Task 7 locally and lands the TF-IDF table in Cloud Storage and BigQuery.

**Before you start:** `submission/phase3_mapreduce.json` and your filled `spark_tfidf.py` must be committed and pushed — the notebook clones your repo and runs what is on `main`. Run the cells **in order**: the report cell at the end collects variables from every section.

## 0. Setup — clone your repo and install dependencies

Colab starts empty, so pull in your repository — that is where your `spark_tfidf.py` and the MapReduce result live. `pyspark` is the only heavy install; the Google client libraries are for sections 5–6.

In [ ]:
# Edit the URL to YOUR repo — the notebook imports your code from it.
# First run: clone. Later runs (after you pushed a fix): pull instead, so the clone
# is refreshed rather than silently left stale. Restart the session afterwards —
# Python keeps already-imported modules in memory (TASKS.md Task 8).
REPO_URL = 'https://github.com/<you>/<your-repo>.git'
%cd /content
import os, subprocess
if os.path.isdir('repo'):
    subprocess.run(['git', '-C', 'repo', 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, 'repo'], check=True)
%cd repo
!pip -q install pyspark pyarrow google-cloud-storage google-cloud-bigquery
# Phase-3 modules are in a subfolder, so add it to the import path.
import sys; sys.path.insert(0, 'phase-3-mapreduce-spark')

### Imports and the corpus

- `corpus.load_documents()` — the same deterministic 60-document TinyShakespeare split that `run_mr.py` used, so the two halves of the phase index the *same* documents.
- `mapreduce` — your primitives, used here only as the local reference.
- `spark_tfidf` — your two RDD stages; `report` — the writer for section 7.

**What to look for:** `documents: 60`.

In [ ]:
import json, time
import mapreduce, corpus, spark_tfidf, report

docs = corpus.load_documents()   # deterministic: TinyShakespeare split into documents
print('documents:', len(docs))

## 1. Your cloud MapReduce result (Lecture 5)

`run_mr.py` wrote `submission/phase3_mapreduce.json` when your Workflows job finished: the execution name, one record per map and reduce task, the elapsed time, and the top terms. This cell loads it and runs the **local reference** — the same `map_wc`, `shuffle`, `reduce_wc` in one process — so you can put the two side by side.

**What to look for:** the top terms must be *identical* — the cell ends with an `OK` line when they are; a `MapReduce result mismatch` error means your deployment and your local code disagree (did you edit `mapreduce.py` after running the cloud job?). Then compare the two elapsed times: the cloud job is **orders of magnitude slower** on this corpus. That is not a bug — it is per-task overhead (function invocations, Cloud Storage round trips, workflow scheduling) on a job far too small to amortise it.

In [ ]:
MR_PATH = 'submission/phase3_mapreduce.json'
try:
    mr_cloud = json.load(open(MR_PATH))
except FileNotFoundError:
    raise SystemExit(f'{MR_PATH} not found in your clone — run run_mr.py in Cloud Shell '
                     '(TASKS.md Task 6), commit the file, push, and re-run section 0.')

# Local reference: same primitives, one process, no network.
t0 = time.perf_counter()
counts = mapreduce.word_count(docs)
local_s = time.perf_counter() - t0
top = report.top_terms(counts, n=10)

print(f"cloud job : {mr_cloud['num_splits']} map tasks, {mr_cloud['num_reducers']} reduce tasks, "
      f"{mr_cloud['cloud_elapsed_s']} s, state {mr_cloud['state']}")
print(f'local run : {local_s*1000:.0f} ms')
print('vocab size:', len(counts), '| top terms:', top)
assert [list(x) for x in mr_cloud['top_terms']] == top, 'MapReduce result mismatch: cloud vs local'
print('OK: cloud job and local reference agree on the top terms')

## 2. PySpark TF-IDF (Lecture 6)

The same corpus, now as a chain of RDD **transformations** (`flatMap`, `reduceByKey`, `join`, `map`) ending in one **action** (`collect`). Nothing computes until that action runs — the earlier lines only build the lineage graph. This section separates the two on purpose: first build the graph and *look at it*, then run it.

### 2a. Start Spark

`local[*]` means one JVM using all Colab cores; the cell is slow because it starts that JVM. Note the `addPyFile` lines: your lambdas call `tokenize` from `mapreduce.py`, and they execute in Spark's **worker processes**, not in this notebook's process. The workers need a copy of the module — this is how code reaches the data in any Spark job, and the reason the driver's `sys.path` alone is not enough.

In [ ]:
from pyspark.sql import SparkSession

# One local Spark: driver + executors inside this Colab VM. setLogLevel hides the noise.
spark = SparkSession.builder.master('local[*]').appName('tfidf').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# Ship your modules to the Python WORKER processes. The driver found them through the
# sys.path edit in section 0, but Spark runs your lambdas in separate worker processes
# (other machines, on a real cluster) that know nothing about that. Without this line the
# first task dies with ModuleNotFoundError: No module named 'mapreduce'.
for f in ('mapreduce.py', 'spark_tfidf.py'):
    spark.sparkContext.addPyFile(f'phase-3-mapreduce-spark/{f}')
print('Spark', spark.version, '| UI at', spark.sparkContext.uiWebUrl)

### 2b. Build the lineage — and read it before anything runs

`build_tfidf_rdd` calls your `term_freq` and `doc_freq`, joins them and maps to rows — and returns instantly, because those are all transformations. `toDebugString()` prints the **lineage graph** Spark has recorded: every RDD, and which one it was derived from.

**How to read it:** the tree is printed bottom-up (the source `ParallelCollectionRDD` is at the bottom). Each `+-` step to the *left* is a **shuffle boundary** — a `ShuffledRDD` from a `reduceByKey` or `join` — and every shuffle boundary starts a new **stage**. Count them: that is how many stages the job will run, before a single task has executed. The `(N)` in front of each RDD is its number of partitions.

In [ ]:
docs_rdd = spark.sparkContext.parallelize(docs)              # distributes the list; no computation
tfidf_rdd = spark_tfidf.build_tfidf_rdd(docs_rdd, len(docs))  # your term_freq + doc_freq, join, map -> rows

# Nothing has run yet. This is the recorded lineage: RDD -> parent RDD -> ... -> source.
print(tfidf_rdd.toDebugString().decode())

### 2c. Run it — the one action

`collect()` is the action: Spark now turns the lineage into stages and tasks, runs them, and ships the rows back to the driver.

**What to look for:** a few thousand `tf-idf rows` and sample rows like `{'term': ..., 'doc_id': ..., 'tf': 2, 'df': 7, 'idf': 2.15, 'tfidf': 4.3}`. Compare the time with section 1 — Spark's start-up and scheduling cost on a tiny corpus is the same lesson again.

In [ ]:
t0 = time.perf_counter()
rows = tfidf_rdd.collect()        # ACTION: everything above executes now
spark_s = time.perf_counter() - t0
print(f'tf-idf rows: {len(rows)} in {spark_s:.1f} s'); print(rows[:3])

Write the table as **Parquet** — a columnar format BigQuery loads natively (section 6).

In [ ]:
spark_tfidf.save_parquet(rows, 'tfidf.parquet')   # pyarrow, columns term/doc_id/tf/df/idf/tfidf
print('wrote tfidf.parquet')

## 3. Look inside the job

Your cloud MapReduce job had the Workflows execution view: parallel map tasks, a barrier, parallel reduce tasks. Spark keeps the same kind of record of every job it runs — which stages, which tasks, when each ran, how many bytes crossed each shuffle — and exposes it two ways: a web UI on the driver, and a REST API behind it. This section uses both.

### 3a. The task timeline, drawn from Spark's own bookkeeping

This cell asks the REST API for the **last job** (your `collect`), lists its stages, and plots every task as a bar: y = task, x = time, colour = stage.

**What to look for:** bands of tasks running *concurrently* within a stage, and gaps *between* stages — each gap is a shuffle boundary, the same barrier your workflow had between map and reduce, and the point where Spark could recover a lost partition from lineage instead of restarting the job. The table above the plot gives the numbers: shuffle **write** in one stage is the shuffle **read** of the next.

If you re-run 2c and then this cell, some stages come back **SKIPPED** and vanish from the plot: Spark still has their shuffle files and reuses them instead of recomputing — that is lineage-based recovery in action, on purpose rather than after a failure.

In [ ]:
import requests
from datetime import datetime
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Spark's UI and its REST API live on the driver (localhost:4040 unless that port was taken).
api = spark.sparkContext.uiWebUrl + '/api/v1/applications/' + spark.sparkContext.applicationId

# The most recent job is the collect() above; a job = one action.
job = max(requests.get(api + '/jobs').json(), key=lambda j: j['jobId'])
stages = sorted((s for s in requests.get(api + '/stages').json()
                 if s['stageId'] in job['stageIds'] and s['status'] == 'COMPLETE'),
                key=lambda s: s['stageId'])

print(f"job {job['jobId']}: {len(stages)} stages, {job['numTasks']} tasks")
print(f"{'stage':>5}  {'tasks':>5}  {'shuffle write':>13}  {'shuffle read':>12}  name")
for s in stages:
    print(f"{s['stageId']:>5}  {s['numTasks']:>5}  {s['shuffleWriteBytes']/1024:>10.1f} KB"
          f"  {s['shuffleReadBytes']/1024:>9.1f} KB  {s['name']}")

# One bar per task: launch time and duration come from the stage's task list.
parse = lambda ts: datetime.strptime(ts, '%Y-%m-%dT%H:%M:%S.%fGMT')
bars = []                                              # (stage, launch_dt, duration_s)
for s in stages:
    tasks = requests.get(f"{api}/stages/{s['stageId']}/{s['attemptId']}/taskList?length=1000").json()
    bars += [(s['stageId'], parse(t['launchTime']), t['duration'] / 1000) for t in tasks]
t_zero = min(b[1] for b in bars)
bars.sort(key=lambda b: (b[0], b[1]))

colors = plt.cm.tab10.colors
fig, ax = plt.subplots(figsize=(10, 0.3 * len(bars) + 1.5))
for y, (stage, launch, dur) in enumerate(bars):
    ax.barh(y, dur, left=(launch - t_zero).total_seconds(), color=colors[stage % 10])
ax.set_xlabel('seconds since first task launched'); ax.set_ylabel('task')
ax.set_title('Spark task timeline for the TF-IDF job (one bar = one task)')
ax.legend(handles=[Patch(color=colors[st % 10], label=f'stage {st}')
                   for st in sorted({b[0] for b in bars})], loc='lower right')
ax.grid(axis='x', alpha=0.3); plt.show()

### 3b. The Spark UI itself (optional, but look once)

The web UI on the driver is what a Spark engineer watches. Colab can proxy it into a new browser tab. Open **Jobs → your job → DAG Visualization** to see the lineage from 2b drawn as boxes grouped into stages, and **Stages → Event Timeline** for an interactive version of the plot above. (Some in-page links may not survive Colab's proxy; the main pages do.)

In [ ]:
from urllib.parse import urlparse
from google.colab import output

# Forward the driver's UI port to your browser; opens a new tab.
output.serve_kernel_port_as_window(urlparse(spark.sparkContext.uiWebUrl).port)

## 4. Transformations, actions, lineage (writeup)

The writeup is a fill-in template: `phase-3-mapreduce-spark/comparison_template.md`, copied to `submission/phase3_comparison.md` (TASKS.md Task 13). Keep these outputs to hand while you fill it — it asks for:

1. The facts — task counts, the three wall times, the stage count from 2b/3a, the retries from your chaos run — and what they mean: rank and explain the timings (the word count single-process vs cloud is like-for-like; Spark's TF-IDF is more work), read the shuffles off the stage count and the retry rate off the retry count.
2. Which operations are **transformations** and which are **actions**; and, from the lineage printed in 2b and the timeline in 3a, how many stages the job had and at which shuffle Spark could recover a lost partition without recomputing everything.
3. Your cloud MapReduce job mapped onto Hadoop — *JobTracker*, *workers*, *HDFS*, *Partitioner* — where the barrier is, why a map task must be **idempotent** for the retry policy to be safe, and what the `--chaos` run showed.

The section-1 numbers in the template must match `phase3_report.json`, which section 7 writes — so fill the template after running section 7.

## 5. Upload the Parquet to Cloud Storage (public)

Colab is not logged into `gcloud`, so authenticate with your Google account first, then use the Python client. The bucket is the one `run_mr.py` already created and made public — the same `<project>-eurecomgpt` bucket Phase 4 will read from.

In [ ]:
from google.colab import auth
auth.authenticate_user()   # opens a popup to log into your Google account

**Set `PROJECT`** to your Phase-0 project id. The IAM step is idempotent — re-adding the `allUsers` binding is harmless if `run_mr.py` already did it.

**What to look for:** the printed URL downloads in a browser.

In [ ]:
PROJECT = 'REPLACE-with-your-project-id'   # <-- your Phase-0 GCP project id
BUCKET  = f'{PROJECT}-eurecomgpt'

from google.cloud import storage
client = storage.Client(project=PROJECT)
try:
    bucket = client.get_bucket(BUCKET)
except Exception:
    bucket = client.create_bucket(BUCKET, location='US')
# Public read via bucket IAM (works with uniform bucket-level access).
policy = bucket.get_iam_policy(requested_policy_version=3)
policy.bindings.append({'role': 'roles/storage.objectViewer', 'members': {'allUsers'}})
bucket.set_iam_policy(policy)
bucket.blob('tfidf.parquet').upload_from_filename('tfidf.parquet')
PARQUET_URL = f'https://storage.googleapis.com/{BUCKET}/tfidf.parquet'
print('public URL:', PARQUET_URL)

## 6. Load the TF-IDF table into BigQuery and query it

The lakehouse layer: BigQuery loads the Parquet straight from Cloud Storage (no rows pass through this notebook), then a SQL query does retrieval over it — the same query shape Phase 4's chat app will run for every user prompt.

**What to look for:** ten `[term, doc_id, tfidf]` rows, highest tf-idf first — rare, document-specific words (names, places), not common ones.

In [ ]:
from google.cloud import bigquery
bq = bigquery.Client(project=PROJECT)
DATASET = f'{PROJECT}.eurecomgpt'
bq.create_dataset(bigquery.Dataset(DATASET), exists_ok=True)
TABLE = f'{DATASET}.tfidf'

# Server-side load: BigQuery reads the Parquet from GCS itself.
job = bq.load_table_from_uri(
    f'gs://{BUCKET}/tfidf.parquet', TABLE,
    job_config=bigquery.LoadJobConfig(source_format=bigquery.SourceFormat.PARQUET,
                                      write_disposition='WRITE_TRUNCATE'))
job.result()   # blocks until the load finishes

q = f'SELECT term, doc_id, tfidf FROM `{TABLE}` ORDER BY tfidf DESC LIMIT 10'
top_by_tfidf = [[r.term, r.doc_id, round(r.tfidf, 4)] for r in bq.query(q).result()]
print('top by tf-idf:', top_by_tfidf)

## 7. Write the submission report

Collects everything into `submission/phase3_report.json`: the cloud MapReduce record from section 1 (plus your local timing), the Spark row count and Parquet URL from sections 2 and 5, and the BigQuery table and query result from section 6. A `NameError` here names the section you skipped.

In [ ]:
report.write_report(
    corpus_info={'num_docs': len(docs)},
    mapreduce={**mr_cloud, 'local_elapsed_s': round(local_s, 3)},
    tfidf={'parquet_gcs_url': PARQUET_URL, 'num_rows': len(rows), 'spark_elapsed_s': round(spark_s, 1)},
    bigquery={'table': TABLE, 'top_by_tfidf': top_by_tfidf},
    comparison={'notes': 'see phase3_comparison.md'},
)

Now **commit** `submission/phase3_report.json` (+ your `phase3_comparison.md`) and push. Keep `wordcount.json` and `tfidf.parquet` public in the bucket until you are graded.